Ciao amici, ho sistemato la lambda (togliendola e scrivendo altro, ma guardate se vi piace di più o di meno, poi c'è una f-string che a me però piace ma ditemi voi) e scritto tutto quello che mi sembrava necessario per capire cosa fa questo codice. Dovrei aver caricato anche il file in csv delle nascite del 2014, almeno si vede anche cosa stampa questo codice. 


In [7]:
import csv #carica il modulo della libreria standard per leggere e scrivere i csv.

def leggi_file(percorso):
    with open(percorso, newline="", encoding="utf-8") as f: #open apre il file, newline="" evita problemi con gli a capo, encoding utf-8 legge le lettere accentate.
        return list(csv.DictReader(f, delimiter=";")) #usa la prima riga come nomi delle colonne e ; come separatore. restituisce un dizionario.
                    #DictReader legge il file riga per riga e trasforma ogni riga in un dizionario. usa prima riga del file (intestazione) come key.
                    #il file nostro è fatto così: 
                    #Anno; Codice Istat; Denominazione; Stato di nascita; ...
                    #2014, 001001; Agliè; Albania; ....
                    #DictReader legge intestazione e per ogni riga successiva produce un dizionario che associa il nome di ogni colonna al valore di quella riga:
                    #{"Anno": "2014", "Codice Istat": "001001", "Denominazione": "Agliè", "Stato di nascita": "Albania", ...}
                    #DictReader dà una riga alla volta, solo quando richieste. Con list le chiediamo tutte e le raccogliamo in una lista. 
                    #la lista che otteniamo è l'elemento righe che passiamo alla def per_comune.
def per_comune(righe, stato):
    stato = stato.strip().lower() #toglie spazi e mette minuscolo
    risultato = {} #fa dizionario da riempire alla fine
    for r in righe: #scorre ogni riga r della lista righe
        if r["Stato di nascita"].strip().lower() != stato: #"Stato di nascita" chiave del dizionario 
            continue #se lo stato è diverso da quello cercato salta alla riga successiva, finchè non le ha scorse tutte
        codice = r["Codice Istat"] #codice ISTAT del comune è univoco, denominazione potrebbe ripetersi
        if codice not in risultato:
            risultato[codice] = {"comune": r["Denominazione"], "maschi": 0, "femmine": 0, "totale": 0}
        risultato[codice]["maschi"] += int(r["Maschi"])
        risultato[codice]["femmine"] += int(r["Femmine"])
        risultato[codice]["totale"] += int(r["Totale"]) #converte i valori da testo a numero e li somma al contatore, += perchè lo stesso comune può comparire in più righe per lo stesso stato
    return risultato
# il risultato che restituisce è un dizionario completo, nella forma {"001001":{"comune":"Agliè", "maschi": 2, "femmine": 3, "totale": 5}, ...}
def stampa(risultato): #risultato è il dizionario di cui sopra
    def prendi_totale(coppia): #è una funzione definita dentro stampa: esiste solo se stampa è in esecuzione, e serve solo per sorted
        return coppia[1]["totale"] #prendi_totale riceve una coppia (codice, dati) e restituisce il totale del comune. Per esempio, con la coppia di Torino:
                    #coppia = ("001272", {"comune": "Torino", "maschi": 30, "femmine": 25, "totale": 55})

                    #coppia[0]              # "001272"  il codice
                    #coppia[1]              # il dizionario con i dati di Torino
                    #coppia[1]["totale"]    # 55
    ordinati = sorted(risultato.items(), key=prendi_totale, reverse=True)
    #risultato.items() trasforma il dizionario in coppie (codcie, dati): ("001001",{... "totale":1}); ("001272", {... "totale": 55}); etc. 
    #key=prendi_totale dice a sorted come decidere l'ordine. sorted chiama prendi_totale su ogni coppia e ottiene 1, 55, etc, poi ordina sulla base dei numeri le coppie.
    #prendi_totale è senza parentesi perchè non va chiamata subito ma va consegnata a sorted.
    #reverse=True inverte l'ordine, dal più grande al più piccolo.
    for codice, d in ordinati: #scorre ordinati e la spacchetta in 2 variabili: codice, che prende il primo elemento, e d il dizionario dei dati.
                               #ordinati è una lista di coppie: ordinati = [("001272", {"comune": "Torino", "maschi": 30, "femmine": 25, "totale": 55}), etc. ]
        print(f'{d["comune"]} ({codice}): {d["totale"]}  (M {d["maschi"]}, F {d["femmine"]})')
        #stampa una riga per comune. La stringa che inizia con f' è una f-string: ciò che sta tra graffe viene sostituito dal valore.
        #per Torino ad esempio stampa: Torino (001272): 55 (M30, F25)
    print("Totale generale:", sum(d["totale"] for d in risultato.values())) #risultato.values dà solo i valori di un dizionario, senza chiavi.
                                    #d è {"comune": "Agliè", ..., "totale": 1}, quindi d["totale"] = 1
                                    #il secondo d sarà 55, il terzo altro valore ancora, con sum avrò la somma dei numeri.
def salva(risultato, percorso): #riceve il dizionario da salvare (risultato) e il nome del file da creare (percorso).
    ordinati = sorted(risultato.items(), key=prendi_totale, reverse=True) #serve perchè il file venga scritto già ordinato.
    with open(percorso, "w", newline="", encoding="utf-8") as f: #apre il file per scriverci dentro e lo chiama f
              #percorso è il nome del file, "w" sta per write, newline="" evita che su windows compaiano righe vuote in più tra una e l'altra riga.
              #encoding per accenti. with chiude il file alla fine del blocco, anche se qualcosa va storto.
        w = csv.writer(f, delimiter=";") #crea l'oggetto che sa scrivere righe csv dentro f. w è variabile che potrebbe assumere qualunque nome, ; come delimitatore.
        w.writerow(["Codice Istat", "Comune", "Maschi", "Femmine", "Totale"]) #scrive l'intestazione, la prima riga del file. Riceve una lista e la trasforma in una riga con gli elementi separati da ;
        for codice, d in ordinati: #scorre le coppie spacchettandole in codic e d.
            w.writerow([codice, d["comune"], d["maschi"], d["femmine"], d["totale"]]) #per ogni comune scrive una riga, costruendo una lista con i cinque valori nello stesso ordine dell'intestazione.

righe = leggi_file("Dati_nascita_2014.csv")

In [ ]:
siria_2014 = per_comune(righe, "Siria")
stampa(siria_2014)
#salva(siria_2014, "siria_per_comune_2014.csv"): ha senso secondo voi salvare ogni volta in un csv a parte? Forse per tokenizzare o mappare i dati dopo? non so


Italia (IT): 5449  (M 3474, F 1975)
Nord-ovest (1): 2280  (M 1433, F 847)
Lombardia (03): 2018  (M 1271, F 747)
Centro (3): 1492  (M 954, F 538)
Nord-est (2): 1225  (M 788, F 437)
Milano (015): 987  (M 618, F 369)
Lazio (12): 954  (M 626, F 328)
Roma (058): 850  (M 553, F 297)
Roma (058091): 647  (M 428, F 219)
Veneto (05): 553  (M 350, F 203)
Emilia-Romagna (08): 500  (M 326, F 174)
Milano (015146): 370  (M 247, F 123)
Sud (4): 316  (M 206, F 110)
Toscana (09): 283  (M 173, F 110)
Como (013): 259  (M 177, F 82)
Padova (028): 229  (M 140, F 89)
Cologno Monzese (015081): 185  (M 120, F 65)
Varese (012): 177  (M 110, F 67)
Marche (11): 151  (M 88, F 63)
Piemonte (01): 150  (M 96, F 54)
Isole (5): 136  (M 93, F 43)
Bergamo (016): 134  (M 80, F 54)
Padova (028060): 132  (M 86, F 46)
Monza e della Brianza (108): 131  (M 88, F 43)
Firenze (048): 127  (M 74, F 53)
Bologna (037): 125  (M 82, F 43)
Venezia (027): 123  (M 82, F 41)
Parma (034): 114  (M 69, F 45)
Campania (15): 111  (M 68, F 43)


Aggiungo anche il resto dei paesi per il 2014 e metto anche quelli del 2015, il motivo per cui non mi funzionava l'altro giorno è che invece di dargli righe_2015, 2016, etc, continuavo a dargli righe e basta e quendi leggeva i dati del 2014.

In [13]:
afghanistan_2014 = per_comune(righe, "Afghanistan")
stampa(afghanistan_2014)
#salva(afghanistan_2014, "afghanistan_per_comune_2014.csv")

Italia (IT): 5654  (M 5238, F 416)
Centro (3): 2793  (M 2664, F 129)
Lazio (12): 2240  (M 2182, F 58)
Roma (058): 2110  (M 2064, F 46)
Roma (058091): 2067  (M 2023, F 44)
Nord-est (2): 856  (M 744, F 112)
Nord-ovest (1): 845  (M 728, F 117)
Sud (4): 795  (M 759, F 36)
Lombardia (03): 511  (M 419, F 92)
Calabria (18): 386  (M 375, F 11)
Isole (5): 365  (M 343, F 22)
Marche (11): 336  (M 295, F 41)
Sicilia (19): 335  (M 315, F 20)
Puglia (16): 324  (M 307, F 17)
Crotone (101): 320  (M 320, F 0)
Piemonte (01): 297  (M 275, F 22)
Emilia-Romagna (08): 295  (M 265, F 30)
Isola di Capo Rizzuto (101013): 288  (M 288, F 0)
Milano (015): 261  (M 241, F 20)
Veneto (05): 203  (M 178, F 25)
Trentino-Alto Adige/Südtirol (04): 194  (M 156, F 38)
Milano (015146): 191  (M 177, F 14)
Torino (001): 186  (M 176, F 10)
Friuli-Venezia Giulia (06): 164  (M 145, F 19)
Bolzano/Bozen (021): 159  (M 125, F 34)
Macerata (043): 159  (M 132, F 27)
Caltanissetta (085): 137  (M 137, F 0)
Torino (001272): 136  (M 130,

In [14]:
iraq_2014 = per_comune(righe, "Iraq")
stampa(iraq_2014)
#salva(iraq_2014, "iraq_per_comune_2014.csv")

Italia (IT): 3390  (M 2435, F 955)
Centro (3): 1211  (M 924, F 287)
Lazio (12): 873  (M 696, F 177)
Roma (058): 829  (M 664, F 165)
Nord-est (2): 823  (M 533, F 290)
Roma (058091): 762  (M 624, F 138)
Sud (4): 645  (M 507, F 138)
Nord-ovest (1): 618  (M 400, F 218)
Lombardia (03): 362  (M 219, F 143)
Trentino-Alto Adige/Südtirol (04): 316  (M 182, F 134)
Calabria (18): 304  (M 247, F 57)
Bolzano/Bozen (021): 300  (M 172, F 128)
Emilia-Romagna (08): 220  (M 146, F 74)
Toscana (09): 212  (M 140, F 72)
Veneto (05): 201  (M 140, F 61)
Bolzano/Bozen (021008): 196  (M 113, F 83)
Puglia (16): 193  (M 164, F 29)
Crotone (101): 191  (M 183, F 8)
Piemonte (01): 189  (M 137, F 52)
Milano (015): 178  (M 111, F 67)
Isola di Capo Rizzuto (101013): 152  (M 149, F 3)
Torino (001): 127  (M 98, F 29)
Venezia (027): 123  (M 87, F 36)
Firenze (048): 121  (M 86, F 35)
Milano (015146): 105  (M 66, F 39)
Firenze (048017): 104  (M 75, F 29)
Torino (001272): 93  (M 73, F 20)
Isole (5): 93  (M 71, F 22)
Friuli-

In [15]:
sudan_2014 = per_comune(righe, "Sudan")
stampa(sudan_2014)
#salva(sudan_2014, "sudan_per_comune.csv")

Italia (IT): 2531  (M 1926, F 605)
Centro (3): 800  (M 608, F 192)
Lazio (12): 640  (M 497, F 143)
Nord-ovest (1): 613  (M 441, F 172)
Roma (058): 601  (M 470, F 131)
Roma (058091): 565  (M 452, F 113)
Nord-est (2): 490  (M 353, F 137)
Lombardia (03): 388  (M 268, F 120)
Sud (4): 357  (M 283, F 74)
Isole (5): 271  (M 241, F 30)
Sicilia (19): 261  (M 234, F 27)
Emilia-Romagna (08): 230  (M 159, F 71)
Piemonte (01): 176  (M 140, F 36)
Veneto (05): 147  (M 110, F 37)
Milano (015): 138  (M 105, F 33)
Torino (001): 135  (M 113, F 22)
Campania (15): 125  (M 105, F 20)
Torino (001272): 107  (M 96, F 11)
Siracusa (089): 106  (M 100, F 6)
Puglia (16): 104  (M 78, F 26)
Milano (015146): 100  (M 78, F 22)
Toscana (09): 92  (M 62, F 30)
Friuli-Venezia Giulia (06): 88  (M 65, F 23)
Siracusa (089017): 86  (M 80, F 6)
Calabria (18): 77  (M 60, F 17)
Parma (034): 75  (M 50, F 25)
Bari (072): 57  (M 37, F 20)
Parma (034027): 56  (M 37, F 19)
Caserta (061): 56  (M 45, F 11)
Napoli (063): 50  (M 43, F 7)

In [16]:
libia_2014 = per_comune(righe, "Libia")
stampa(libia_2014) 
#salva(libia_2014, "libia_per_comune_2014.csv")

Italia (IT): 34361  (M 16477, F 17884)
Centro (3): 12491  (M 5966, F 6525)
Lazio (12): 9075  (M 4304, F 4771)
Nord-ovest (1): 8485  (M 4132, F 4353)
Roma (058): 7228  (M 3378, F 3850)
Roma (058091): 5729  (M 2618, F 3111)
Nord-est (2): 5556  (M 2758, F 2798)
Isole (5): 4407  (M 2000, F 2407)
Lombardia (03): 4360  (M 2129, F 2231)
Sicilia (19): 4098  (M 1850, F 2248)
Sud (4): 3422  (M 1621, F 1801)
Piemonte (01): 3144  (M 1530, F 1614)
Veneto (05): 2765  (M 1333, F 1432)
Toscana (09): 2401  (M 1141, F 1260)
Torino (001): 2128  (M 1021, F 1107)
Emilia-Romagna (08): 2122  (M 1088, F 1034)
Milano (015): 1780  (M 883, F 897)
Latina (059): 1349  (M 669, F 680)
Siracusa (089): 1270  (M 576, F 694)
Campania (15): 1270  (M 591, F 679)
Torino (001272): 1044  (M 477, F 567)
Catania (087): 1037  (M 453, F 584)
Liguria (07): 966  (M 467, F 499)
Abruzzo (13): 904  (M 432, F 472)
Siracusa (089017): 890  (M 401, F 489)
Puglia (16): 856  (M 401, F 455)
Milano (015146): 773  (M 383, F 390)
Latina (05901

Guardiamo adesso il 2015 (2016, 2003, 2025 si fanno tutti allo stesso modo, per ora metto solo questi 2 anni)

In [17]:
righe_2015 = leggi_file("Dati_nascita_2015.csv")

In [18]:
siria_2015 = per_comune(righe_2015, "Siria")
stampa(siria_2015)
#salva(siria_2015, "siria_per_comune_2015.csv")

Italia (IT): 5704  (M 3618, F 2086)
Nord-ovest (1): 2347  (M 1454, F 893)
Lombardia (03): 2067  (M 1280, F 787)
Centro (3): 1557  (M 1004, F 553)
Nord-est (2): 1297  (M 824, F 473)
Lazio (12): 1004  (M 669, F 335)
Milano (015): 1000  (M 619, F 381)
Roma (058): 901  (M 598, F 303)
Roma (058091): 696  (M 471, F 225)
Veneto (05): 592  (M 372, F 220)
Emilia-Romagna (08): 517  (M 336, F 181)
Milano (015146): 384  (M 249, F 135)
Sud (4): 332  (M 221, F 111)
Toscana (09): 304  (M 186, F 118)
Como (013): 274  (M 180, F 94)
Padova (028): 251  (M 155, F 96)
Cologno Monzese (015081): 195  (M 128, F 67)
Varese (012): 181  (M 109, F 72)
Isole (5): 171  (M 115, F 56)
Piemonte (01): 162  (M 104, F 58)
Monza e della Brianza (108): 149  (M 95, F 54)
Padova (028060): 144  (M 91, F 53)
Marche (11): 141  (M 80, F 61)
Venezia (027): 134  (M 87, F 47)
Firenze (048): 134  (M 78, F 56)
Bologna (037): 133  (M 87, F 46)
Bergamo (016): 130  (M 78, F 52)
Trentino-Alto Adige/Südtirol (04): 126  (M 83, F 43)
Erba (

In [19]:
afghanistan_2015 = per_comune(righe_2015, "Afghanistan")
stampa(afghanistan_2015)
#salva(afghanistan_2015, "afghanistan_per_comune_2015.csv")

Italia (IT): 6513  (M 6045, F 468)
Centro (3): 2974  (M 2822, F 152)
Lazio (12): 2307  (M 2235, F 72)
Roma (058): 2139  (M 2081, F 58)
Roma (058091): 2073  (M 2018, F 55)
Nord-est (2): 1086  (M 956, F 130)
Sud (4): 1086  (M 1046, F 40)
Nord-ovest (1): 920  (M 791, F 129)
Lombardia (03): 565  (M 463, F 102)
Puglia (16): 474  (M 455, F 19)
Calabria (18): 472  (M 461, F 11)
Isole (5): 447  (M 430, F 17)
Sicilia (19): 417  (M 401, F 16)
Marche (11): 379  (M 336, F 43)
Crotone (101): 378  (M 378, F 0)
Emilia-Romagna (08): 347  (M 311, F 36)
Milano (015): 304  (M 279, F 25)
Piemonte (01): 303  (M 282, F 21)
Isola di Capo Rizzuto (101013): 279  (M 279, F 0)
Trentino-Alto Adige/Südtirol (04): 277  (M 234, F 43)
Veneto (05): 242  (M 208, F 34)
Bolzano/Bozen (021): 228  (M 193, F 35)
Friuli-Venezia Giulia (06): 220  (M 203, F 17)
Caltanissetta (085): 217  (M 216, F 1)
Milano (015146): 209  (M 192, F 17)
Toscana (09): 200  (M 179, F 21)
Macerata (043): 189  (M 161, F 28)
Caltanissetta (085004): 1

In [20]:
iraq_2015 = per_comune(righe_2015, "Iraq")
stampa(iraq_2015)
#salva(iraq_2015, "irqa_per_comune_2015.csv")

Italia (IT): 3493  (M 2526, F 967)
Centro (3): 1238  (M 939, F 299)
Lazio (12): 895  (M 704, F 191)
Roma (058): 846  (M 668, F 178)
Nord-est (2): 820  (M 520, F 300)
Roma (058091): 781  (M 629, F 152)
Sud (4): 738  (M 611, F 127)
Nord-ovest (1): 608  (M 389, F 219)
Lombardia (03): 361  (M 219, F 142)
Trentino-Alto Adige/Südtirol (04): 326  (M 187, F 139)
Bolzano/Bozen (021): 311  (M 177, F 134)
Puglia (16): 291  (M 262, F 29)
Calabria (18): 287  (M 242, F 45)
Emilia-Romagna (08): 217  (M 144, F 73)
Toscana (09): 214  (M 142, F 72)
Bolzano/Bozen (021008): 204  (M 117, F 87)
Veneto (05): 197  (M 131, F 66)
Crotone (101): 192  (M 181, F 11)
Piemonte (01): 187  (M 128, F 59)
Milano (015): 186  (M 122, F 64)
Isola di Capo Rizzuto (101013): 143  (M 140, F 3)
Torino (001): 131  (M 97, F 34)
Firenze (048): 124  (M 89, F 35)
Venezia (027): 122  (M 82, F 40)
Milano (015146): 112  (M 73, F 39)
Bari (072): 111  (M 95, F 16)
Firenze (048017): 103  (M 76, F 27)
Torino (001272): 99  (M 74, F 25)
Fogg

In [21]:
sudan_2015 = per_comune(righe_2015, "Sudan")
stampa(sudan_2015)
#salva(sudan_2015, "sudan_per_comune_2015.csv")

Italia (IT): 2550  (M 1924, F 626)
Centro (3): 806  (M 604, F 202)
Lazio (12): 639  (M 488, F 151)
Nord-ovest (1): 632  (M 449, F 183)
Roma (058): 602  (M 461, F 141)
Roma (058091): 566  (M 443, F 123)
Nord-est (2): 460  (M 324, F 136)
Lombardia (03): 382  (M 255, F 127)
Sud (4): 356  (M 279, F 77)
Isole (5): 296  (M 268, F 28)
Sicilia (19): 278  (M 253, F 25)
Emilia-Romagna (08): 233  (M 159, F 74)
Piemonte (01): 203  (M 162, F 41)
Torino (001): 165  (M 135, F 30)
Torino (001272): 140  (M 120, F 20)
Milano (015): 135  (M 100, F 35)
Veneto (05): 127  (M 92, F 35)
Campania (15): 120  (M 99, F 21)
Siracusa (089): 112  (M 106, F 6)
Puglia (16): 105  (M 78, F 27)
Toscana (09): 100  (M 69, F 31)
Milano (015146): 99  (M 75, F 24)
Siracusa (089017): 89  (M 84, F 5)
Friuli-Venezia Giulia (06): 78  (M 57, F 21)
Calabria (18): 78  (M 59, F 19)
Parma (034): 77  (M 52, F 25)
Caserta (061): 58  (M 45, F 13)
Parma (034027): 56  (M 36, F 20)
Bari (072): 52  (M 33, F 19)
Bologna (037): 45  (M 27, F 18

In [22]:
libia_2015 = per_comune(righe_2015, "Libia")
stampa(libia_2015)
#salva(libia_2015, "libia_per_comune_2015.csv")

Italia (IT): 33960  (M 16265, F 17695)
Centro (3): 12352  (M 5902, F 6450)
Lazio (12): 8997  (M 4274, F 4723)
Nord-ovest (1): 8406  (M 4090, F 4316)
Roma (058): 7175  (M 3362, F 3813)
Roma (058091): 5681  (M 2600, F 3081)
Nord-est (2): 5504  (M 2722, F 2782)
Isole (5): 4337  (M 1968, F 2369)
Lombardia (03): 4335  (M 2120, F 2215)
Sicilia (19): 4028  (M 1817, F 2211)
Sud (4): 3361  (M 1583, F 1778)
Piemonte (01): 3112  (M 1513, F 1599)
Veneto (05): 2741  (M 1316, F 1425)
Toscana (09): 2365  (M 1125, F 1240)
Torino (001): 2112  (M 1012, F 1100)
Emilia-Romagna (08): 2102  (M 1076, F 1026)
Milano (015): 1780  (M 885, F 895)
Latina (059): 1333  (M 658, F 675)
Siracusa (089): 1240  (M 559, F 681)
Campania (15): 1239  (M 569, F 670)
Torino (001272): 1033  (M 471, F 562)
Catania (087): 1015  (M 442, F 573)
Liguria (07): 944  (M 452, F 492)
Abruzzo (13): 894  (M 423, F 471)
Siracusa (089017): 859  (M 387, F 472)
Puglia (16): 837  (M 395, F 442)
Milano (015146): 762  (M 377, F 385)
Latina (05901

Questo è il codice per le medie con cui poi fare i confronti. Anche qui vale la stessa logica per cui sono gli stessi elementi del codice ma cambia solo ciò che si vuole analizzare. Il problema qua era capire come mai calcola con questi maggiori o minori, ma rientra tra le cose che devo ancora vedere :').


In [ ]:

FILES = {
    "2014": "Dati_nascita_2014.csv",   
    "2015": "Dati_nascita_2015.csv",
    "2016": "Dati_nascita_2016.csv",
}

ANNI = ["2014", "2015", "2016"]
STATI = ["Siria", "Afghanistan", "Iraq", "Sudan", "Libia"]   

def totale_stato_anno(percorso, stato, anno):
    stato = stato.strip().lower()
    totale = 0
    with open(percorso, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f, delimiter=";"):
            if r["Anno"].strip() == anno and r["Stato di nascita"].strip().lower() == stato:
                totale += int(r["Totale"])
    return totale

print(f'{"Stato":<15}{"2014":>8}{"2015":>8}{"2016":>8}{"Media diff/anno":>18}{"Media var %/anno":>19}')

medie_variazioni = {}
for stato in STATI:
    valori = [totale_stato_anno(FILES[a], stato, a) for a in ANNI]
    diffs = [valori[i + 1] - valori[i] for i in range(len(valori) - 1)]
    percs = [
        (valori[i + 1] - valori[i]) / valori[i] * 100
        for i in range(len(valori) - 1)
        if valori[i] > 0
    ]
    media_diff = sum(diffs) / len(diffs)
    media_perc = sum(percs) / len(percs) if percs else None
    medie_variazioni[stato] = {"valori": valori, "media_diff": media_diff, "media_perc": media_perc}

    perc_txt = f"{media_perc:.1f}" if media_perc is not None else "n.d."
    print(f"{stato:<15}{valori[0]:>8}{valori[1]:>8}{valori[2]:>8}{media_diff:>+18.1f}{perc_txt:>19}")